# 02. Startup Scenario and Vacuum Fields

This session follows one VEST startup in the order an experimentalist encounters it:

**response → breakdown time → actuators → vessel circuit → validated vacuum field → startup physics.**

The default path uses packaged shot 39915 and runs offline. The same public APIs can be reused
with `vaft.database.load(...)` for arbitrary VEST discharges when the mapped diagnostics exist.


## Session goals

By the end of the session you should be able to:

- identify plasma formation from current, light, diamagnetic, loop-voltage and local magnetic signals;
- use one detected `t_breakdown` consistently instead of choosing analysis times by eye;
- connect TF/PF/gas settings to the vacuum field present before a plasma equilibrium exists;
- understand how axisymmetric Green functions generate mutual-inductance matrices, vessel eddy currents,
  synthetic magnetics and vacuum-field maps;
- use \(V_{\rm loop}\), reference-point \(B_z\), decay index and Lloyd/Townsend quantities as **reduced
  startup proxies**, without mistaking them for a reconstructed equilibrium;
- compare several discharges on both absolute and breakdown-aligned time bases.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import vaft

from vaft.formula.ecr import electron_cyclotron_resonance_field
from vaft.omas.plasma_timing import plasma_timing
from vaft.omas.discharge_timing import discharge_timing

ods = vaft.omas.sample_ods()
shot = (
    int(np.asarray(ods["dataset_description.data_entry.pulse"]).ravel()[0])
    if "dataset_description.data_entry.pulse" in ods else 39915
)
print(f"Using packaged sample shot {shot}.")

# Lab/database path: replace the packaged sample with any mapped discharge.
# shot = 45531
# ods = vaft.database.load(shot)


## 1. Observe the discharge before modelling it

Start with what the machine measured. Plot each family separately before combining anything:
plasma current, H-alpha, impurity line radiation, diamagnetic flux, an inboard-midplane flux-loop
voltage estimate, and a near-midplane inboard \(B_z\)-sensitive probe.

The packaged 39915 sample stores flux-loop **flux** but not `voltage.data`. For that offline case the
cell below displays \(-d\Phi/dt\) from the stored flux. A database-loaded ODS that contains
`magnetics.flux_loop.<i>.voltage.data` instead uses the canonical voltage renderer. The sign and
full-weber/per-radian convention remain part of issue #354, so this diagnostic derivative is shown
as a convention-aware observable rather than silently equated to every other loop-voltage kernel.


In [ ]:
# Plasma current
vaft.omas.plot_plasma_current_time(ods)
plt.show()

# Hydrogen light
vaft.omas.plot_spectrometer_uv_time_intensity(ods, emission="H_alpha")
plt.show()

# All mapped carbon and oxygen impurity lines together, rather than one hard-coded CIII channel.
vaft.omas.plot_spectrometer_uv_time_intensity(
    ods, emission=["C", "O"], layout="subplots"
)
plt.show()

# Diamagnetic flux
vaft.omas.plot_diamagnetic_flux_time(ods)
plt.show()

# Identify the inboard-midplane flux loop from mapped geometry.
flat_paths = list(ods.flat())
flux_loop_ids = sorted({
    int(path.split(".")[2])
    for path in flat_paths
    if path.startswith("magnetics.flux_loop.") and path.endswith(".position.0.r")
})
flux_loop_points = np.array([
    [
        float(ods[f"magnetics.flux_loop.{i}.position.0.r"]),
        float(ods[f"magnetics.flux_loop.{i}.position.0.z"]),
    ]
    for i in flux_loop_ids
])
inboard_loop_candidates = np.where(
    flux_loop_points[:, 0] <= np.nanmedian(flux_loop_points[:, 0])
)[0]
inboard_loop_local = inboard_loop_candidates[
    np.nanargmin(np.abs(flux_loop_points[inboard_loop_candidates, 1]))
]
inboard_flux_loop_idx = flux_loop_ids[int(inboard_loop_local)]

voltage_path = f"magnetics.flux_loop.{inboard_flux_loop_idx}.voltage.data"
if voltage_path in flat_paths:
    vaft.omas.plot_flux_loop_time_voltage(
        ods, selection=[inboard_flux_loop_idx]
    )
    plt.show()
    diagnostic_vloop_time = np.asarray(ods["magnetics.time"], dtype=float)
    diagnostic_vloop = np.asarray(ods[voltage_path], dtype=float)
else:
    diagnostic_vloop_time = np.asarray(ods["magnetics.time"], dtype=float)
    loop_flux = np.asarray(
        ods[f"magnetics.flux_loop.{inboard_flux_loop_idx}.flux.data"], dtype=float
    )
    diagnostic_vloop = -vaft.process.time_derivative(diagnostic_vloop_time, loop_flux)
    fig, ax = plt.subplots()
    ax.plot(diagnostic_vloop_time, diagnostic_vloop)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel(r"$-d\Phi/dt$ [V]")
    ax.set_title(f"Flux loop {inboard_flux_loop_idx}: inboard-midplane voltage estimate")
    ax.grid(alpha=0.3)
    plt.show()

# Identify a near-midplane inboard probe whose mapped orientation is closest to +Bz.
probe_ids = sorted({
    int(path.split(".")[2])
    for path in flat_paths
    if path.startswith("magnetics.b_field_pol_probe.") and path.endswith(".position.r")
})
probe_r = np.array([float(ods[f"magnetics.b_field_pol_probe.{i}.position.r"]) for i in probe_ids])
probe_z = np.array([float(ods[f"magnetics.b_field_pol_probe.{i}.position.z"]) for i in probe_ids])
probe_angle = np.array([
    float(ods[f"magnetics.b_field_pol_probe.{i}.poloidal_angle"])
    if f"magnetics.b_field_pol_probe.{i}.poloidal_angle" in flat_paths else np.nan
    for i in probe_ids
])
inboard_probe_candidates = np.where(probe_r <= np.nanmedian(probe_r))[0]
target_angle = 3.0 * np.pi / 2.0  # IMAS DD convention: +Bz
angle_error = np.abs(np.angle(np.exp(1j * (probe_angle - target_angle))))
vertical_candidates = inboard_probe_candidates[
    np.isfinite(angle_error[inboard_probe_candidates])
    & (angle_error[inboard_probe_candidates] < np.deg2rad(25.0))
]
if vertical_candidates.size == 0:
    vertical_candidates = inboard_probe_candidates
inboard_bz_local = vertical_candidates[np.nanargmin(np.abs(probe_z[vertical_candidates]))]
inboard_bz_idx = probe_ids[int(inboard_bz_local)]

vaft.omas.plot_b_field_probe_time_field(ods, selection=[inboard_bz_idx])
plt.show()


### See the response

The fast camera belongs here, before the circuit model, because it gives an immediate visual meaning
to “startup.” The repository ships a sparse set of calibrated frames for shot 39915. In a live
notebook use `interaction_backend="auto"` to drag the frame slider; CI drives the same control
headlessly.


In [ ]:
import cv2
from vaft.machine_mapping.camera_visible import (
    vfit_camera_visible_dynamic,
    vfit_camera_visible_static,
)

frames = vaft.data.sample_camera_visible_frame_paths(shot)[::11]
images = [cv2.imread(str(path), cv2.IMREAD_GRAYSCALE) for _, path in frames]
frame_times = [time_s for time_s, _ in frames]
if not images or any(image is None for image in images):
    print("Calibrated repository camera frames are unavailable for this shot.")
else:
    vfit_camera_visible_static(
        ods,
        lines_n=images[0].shape[0],
        columns_n=images[0].shape[1],
        channel_name="Fast Camera",
    )
    vfit_camera_visible_dynamic(ods, images=images, times_s=frame_times)
    result = vaft.omas.plot_camera_visible_image(
        ods, interactive=True, interaction_backend="none"
    )
    slider = next(control for control in result.controls if control.name == "frame_index")
    print(f"{slider.label}: {slider.options[1] + 1} sampled frames")
    plt.close(result.figure)


## 2. Fix the event time once

Every later startup plot uses the same `t_breakdown`. `find_breakdown_onset` is the public compact
interface; `plasma_timing` retains the H-alpha/current agreement diagnostics, while
`discharge_timing` supplies actuator events such as the Ohmic-coil onset.


In [ ]:
t_breakdown = float(vaft.omas.find_breakdown_onset(ods))
timing = plasma_timing(ods)
events = discharge_timing(ods)

ip_time = np.asarray(ods["magnetics.ip.0.time"], dtype=float)
ip_data = np.asarray(ods["magnetics.ip.0.data"], dtype=float)
t_ip_peak = float(ip_time[np.nanargmax(np.abs(ip_data))])
t_coil_onset = float(events.oh_onset)

print(f"breakdown onset : {t_breakdown * 1e3:.2f} ms ({timing.source})")
print(f"OH onset        : {t_coil_onset * 1e3:.2f} ms")
print(f"|Ip| peak       : {t_ip_peak * 1e3:.2f} ms")
print(f"light/current   : {timing.agreement}")

# Six-row common response view.
fig, axes = plt.subplots(6, 1, sharex=True, figsize=(10, 13))
vaft.omas.plot_plasma_current_time(ods, ax=axes[0], show=False)
vaft.omas.plot_spectrometer_uv_time_intensity(
    ods, emission="H_alpha", ax=axes[1], show=False
)
vaft.omas.plot_spectrometer_uv_time_intensity(
    ods, emission=["C", "O"], layout="overlay", ax=axes[2], show=False
)
vaft.omas.plot_diamagnetic_flux_time(ods, ax=axes[3], show=False)
axes[4].plot(diagnostic_vloop_time, diagnostic_vloop)
axes[4].set_ylabel(r"$V_{\rm loop}$ proxy [V]")
axes[4].grid(alpha=0.3)
vaft.omas.plot_b_field_probe_time_field(
    ods, selection=[inboard_bz_idx], ax=axes[5], show=False
)
for ax in axes:
    ax.axvline(t_breakdown, color="k", ls="--", lw=0.8)
axes[-1].set_xlabel("Time [s]")
fig.suptitle(f"Shot {shot}: measured response and detected breakdown")
plt.show()


## 3. What drove it?

### TF, PF, gas and the EC resonance assumption

For VEST, the startup interpretation used here is configuration-specific:

- **PF1** is the primary/Ohmic transformer drive coil: it creates loop-voltage drive, not “Ohmic
  power” by itself.
- **PF5** participates in forming the startup null.
- **PF6 and PF9/10** provide much of the vertical/equilibrium-field control in this configuration.

The VEST PF waveform is largely a capacitor-bank discharge: charging voltage, P/S switching times,
capacitance and circuit \(L/R\) determine the waveform. A programmable current-regulated/H-bridge
system instead uses feedforward to request a shaped or flat waveform and feedback to correct the
remaining tracking/shape error.

The EC mapper is still tracked by **#165**. We therefore do **not** invent an `ec_launchers` power
trace. For this tutorial only, the surviving pre-ionisation system is represented by an explicit
**2.45 GHz** frequency assumption, used solely to locate the cold fundamental resonance.


In [ ]:
R_STARTUP_REF = 0.4
EC_FREQUENCY_HZ = 2.45e9
B_ECR = electron_cyclotron_resonance_field(EC_FREQUENCY_HZ)

vaft.omas.plot_tf_coil_time_current(ods)
plt.show()
vaft.omas.plot_tf_coil_time_b_t(ods)
plt.show()

vaft.omas.plot_pf_coil_time_current(ods, layout="subplots")
plt.show()
vaft.omas.plot_pf_coil_time_current_turns(ods, layout="subplots")
plt.show()

vaft.omas.plot_barometry_time_pressure(ods, yunit="Torr")
plt.show()
vaft.omas.plot_barometry_time_pressure(ods, yunit="Pa")
plt.show()

# IMAS tf.b_field_tor_vacuum_r.data is B_T * R [T m], not B_T [T].
tf_time = np.asarray(ods["tf.b_field_tor_vacuum_r.time"], dtype=float)
tf_btR = np.asarray(ods["tf.b_field_tor_vacuum_r.data"], dtype=float)
btR_breakdown = float(np.interp(t_breakdown, tf_time, tf_btR))
R_ECR = abs(btR_breakdown) / B_ECR

print(f"2.45 GHz fundamental: B_ECR = {B_ECR:.5f} T")
print(f"at breakdown: |B_T R| = {abs(btR_breakdown):.5f} T m -> R_ECR = {R_ECR:.3f} m")

fig, ax = plt.subplots()
ax.plot(tf_time, np.abs(tf_btR) / B_ECR)
ax.axvline(t_breakdown, color="k", ls="--", lw=0.8)
ax.axhline(R_ECR, color="0.5", ls=":")
ax.set_xlabel("Time [s]")
ax.set_ylabel(r"$R_{\rm ECR}$ [m]")
ax.set_title("2.45 GHz cold fundamental resonance radius")
ax.grid(alpha=0.3)
plt.show()


## 4. From geometry to mutual inductance to vessel current

Before writing the circuit equation, look at the loops it refers to. Active PF coils are driven;
the conducting vessel/passive structure is represented by many axisymmetric loops. A later
plasma-current representation can be treated by the same filament machinery, but the **vacuum**
solve below deliberately passes no plasma filaments.

For an axisymmetric circular filament,
\[
\psi(R,Z)=G_\psi(R,Z;R',Z')\,I .
\]
The exact Green response contains the complete elliptic integrals of the circular-filament problem.
Evaluating the flux response of source loop \(j\) at loop \(i\) gives the off-diagonal **mutual**
inductance \(M_{ij}\); diagonal terms require the finite-conductor/self-inductance model.

With \(\mathbf I_v\) the passive/vessel currents and \(\mathbf I_a\) active-coil currents,
\[
\mathbf L_{vv}\dot{\mathbf I}_v+\mathbf R_v\mathbf I_v
  =-\mathbf M_{va}\dot{\mathbf I}_a-\mathbf M_{vp}\dot{\mathbf I}_p .
\]
Before breakdown the plasma-drive term is absent. The drive depends on **current change**, which is
why the vessel response lags and decays on its own \(L/R\) time scales.


In [ ]:
vaft.omas.plot_machine_geometry_poloidal(ods)
plt.show()

# Empty plasma filament lists select the vacuum response.
vaft.omas.compute_eddy_currents(ods, [], [])

vaft.omas.plot_current_overview(ods)
plt.show()


### One Green-function machinery, five uses

The same response matrices now support five conceptually different tasks:

1. solve vessel **eddy currents** from the driven PF circuits;
2. predict the coil + eddy contribution seen by magnetic probes and flux loops;
3. add PF, vessel and later plasma-current contributions to field maps used by equilibrium work
   (Tutorial 03);
4. before an equilibrium exists, evaluate vacuum-field startup questions such as null quality,
   \(E_\varphi\), Townsend/Lloyd accessibility and connection length;
5. after current appears, use reference-point \(B_z\), \(V_{\rm loop}\) and decay index as reduced
   operating proxies.

For interpretation only, a large-aspect-ratio current-ring scaling is
\[
B_v \simeq -\frac{\mu_0 I_p}{4\pi R_0}
\left[\ln\frac{8R_0}{a}+\frac{l_i}{2}+\beta_p-\frac{3}{2}\right].
\]
It is a **radial force-balance proxy**, not a replacement for a Grad-Shafranov equilibrium.
Likewise \(V_{\rm loop}\) represents transformer/Ohmic electric-field drive; during current ramp,
\(V_{\rm loop}I_p\) is **not identical to dissipated Ohmic power** because magnetic energy is also
changing. The decay index is a rigid-ring vertical-stability proxy; the conducting wall changes
the true stability window.


## 5. Validate the vacuum model against magnetics

Model-derived startup quantities are only useful after the coil + vessel response reproduces the
measured magnetic channels. First compare measured, coil-only and coil+eddy signals; then inspect
the plasma residual.


In [ ]:
vaft.omas.plot_magnetics_overview_vacuum(ods)
plt.show()


In [ ]:
vaft.omas.plot_magnetics_overview_plasma_residual(ods)
plt.show()


### Rogowski sensors are not the derived quantities

`magnetics.rogowski_coil` stores the calibrated physical sensor currents. The processing chain is:

**raw DAQ → calibration/integration → `magnetics.rogowski_coil[*].current` → compensation /
baseline / sign processing → derived quantity.**

The plasma-current chain ends in `magnetics.ip`; the diamagnetic chain ends in
`magnetics.diamagnetic_flux`. Calling the mapped Rogowski current “raw DAQ voltage” would collapse
two distinct stages and hide exactly the processing students need to understand.


In [ ]:
rogowski_ids = sorted({
    int(path.split(".")[2])
    for path in ods.flat()
    if path.startswith("magnetics.rogowski_coil.") and path.endswith(".current.data")
})
print(f"mapped physical Rogowski sensors: {rogowski_ids}")
for i in rogowski_ids:
    name_path = f"magnetics.rogowski_coil.{i}.name"
    name = str(ods[name_path]) if name_path in list(ods.flat()) else f"rogowski_coil[{i}]"
    current = np.asarray(ods[f"magnetics.rogowski_coil.{i}.current.data"], dtype=float)
    print(f"  {name}: {current.size} calibrated sensor-current samples")
print("derived paths: magnetics.ip.0.data, magnetics.diamagnetic_flux.0.data")


## 6. Startup proxies on a common time base

Use the interval from Ohmic-coil onset to the peak plasma current. The first curve below is the
modelled vacuum loop voltage at the VEST startup reference point
\((R,Z)=(0.4\,{\rm m},0)\). The radial \(B_z\) and decay-index cut is evaluated at the common
breakdown instant; the next implementation step turns these radial cuts into a shared time slider.


In [ ]:
time, v_loop = vaft.omas.compute_startup_loop_voltage_ods(
    ods, rz=(R_STARTUP_REF, 0.0)
)
radius, decay_index = vaft.omas.compute_decay_index_ods(ods, time=t_breakdown)

fig, axes = plt.subplots(2, 1, figsize=(9, 7))
axes[0].plot(time, v_loop)
axes[0].axvline(t_breakdown, color="k", ls="--", lw=0.8)
axes[0].set_xlim(t_coil_onset, t_ip_peak)
axes[0].set_ylabel(r"$V_{\rm loop}$ [V]")
axes[0].grid(alpha=0.3)

axes[1].plot(radius, decay_index)
axes[1].axhspan(0.0, 1.5, alpha=0.15, label="rigid-ring reference band")
axes[1].set_xlabel("R [m]")
axes[1].set_ylabel("decay index n")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.show()

print(f"peak |V_loop| in model trace: {np.nanmax(np.abs(v_loop)):.2f} V")


## 7. Vacuum topology at breakdown

Before a plasma equilibrium exists, the relevant field is the PF + vessel **vacuum** field. The
midplane flux cut shows the null structure, and the 2-D maps expose \(\psi\), \(|B_p|\),
\(E_\varphi\), decay index and the startup-accessibility quantities. The vertical line marks the
2.45 GHz ECR radius inferred from \(B_TR\).


In [ ]:
vaft.omas.compute_vacuum_fields_1d(ods, time=t_breakdown)
fig, ax = vaft.omas.plot_equilibrium_field_psi_vacuum(
    ods,
    times=t_breakdown,
    z=0.0,
    interactive=False,
    interaction_backend="none",
)
ax.axvline(R_ECR, color="0.4", ls="--", label="2.45 GHz ECR")
ax.legend()
plt.show()

fields = ("psi", "b_poloidal", "e_toroidal", "decay_index")
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, quantity in zip(axes, fields):
    vaft.omas.plot_vacuum_field(
        ods, field=quantity, time=t_breakdown, resolution=65, ax=ax, show=False
    )
    ax.axvline(R_ECR, color="w", ls="--", lw=1.0)
    ax.set_title(quantity)
fig.suptitle(f"Vacuum field at breakdown, t = {t_breakdown * 1e3:.2f} ms")
plt.show()


In [ ]:
# In Jupyter, change to interaction_backend="auto" to drag the time slider.
result = vaft.omas.plot_vacuum_field(
    ods,
    field="b_poloidal",
    resolution=33,
    interactive=True,
    interaction_backend="none",
)
slider = next(control for control in result.controls if control.name == "time_index")
print(f"{slider.label}: {slider.options[1] + 1} available time positions")
plt.close(result.figure)


## 8. Pressure, connection length and Lloyd/Townsend accessibility

A weak poloidal field is useful because it lengthens an open electron trajectory. Lloyd's reduced
criterion combines pressure, connection length and toroidal electric field; it is a startup
accessibility model, not a burn-through model.


In [ ]:
pressure = np.asarray(ods["barometry.gauge.0.pressure.data"], dtype=float)
pressure_time = np.asarray(ods["barometry.gauge.0.pressure.time"], dtype=float)
pre_window = pressure_time < t_coil_onset
prefill = float(np.nanmedian(pressure[pre_window]))

limiter_r = np.asarray(ods["wall.description_2d.0.limiter.unit.0.outline.r"], dtype=float)
minor_radius = 0.5 * float(np.nanmax(limiter_r) - np.nanmin(limiter_r))
b_toroidal = abs(btR_breakdown) / R_STARTUP_REF

b_perp = np.geomspace(1e-4, 5e-3, 200)
length_open = minor_radius * b_toroidal / b_perp
e_threshold = vaft.formula.lloyd_breakdown_field(prefill, length_open)
e_drive = float(np.interp(t_breakdown, time, np.abs(v_loop))) / (
    2.0 * np.pi * R_STARTUP_REF
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].loglog(b_perp * 1e4, length_open)
axes[0].set_ylabel(r"$L_{\rm open}$ [m]")
axes[1].loglog(b_perp * 1e4, e_threshold)
axes[1].axhline(e_drive, color="k", ls="--", lw=0.8, label="drive at breakdown")
axes[1].set_ylabel(r"$E_{\rm BD}$ [V/m]")
axes[1].legend()
for ax in axes:
    ax.set_xlabel(r"$B_\perp$ [G]")
    ax.grid(alpha=0.3, which="both")
plt.show()

traced = vaft.omas.compute_connection_length_map_ods(
    ods, time=t_breakdown, resolution=33
)
vaft.omas.plot_connection_length_map(traced)
plt.show()

print(f"prefill = {prefill:.3e} Pa = {prefill / vaft.formula.PA_PER_TORR:.3e} Torr")
print(f"drive at breakdown = {e_drive:.3f} V/m")


## 9. Camera alignment near breakdown

The next target is to project **vacuum** field-line trajectories seeded at the startup reference
point \((R_{\rm ref},0)\) and at \((R_{\rm ECR},0)\) onto the calibrated camera frame nearest
`t_breakdown`. The current public camera `field_line` overlay is equilibrium-oriented, so this
tutorial does not silently substitute an EFIT line before EFIT exists. The early frame slider above
is the measured image; extending the overlay to accept explicit vacuum trajectories is kept as an
implementation boundary rather than faking the result.


In [ ]:
if images:
    onset_frame_index = int(np.argmin(np.abs(np.asarray(frame_times) - t_breakdown)))
    vaft.omas.plot_camera_visible_image(
        ods, frame_index=onset_frame_index, projection="calibrated"
    )
    plt.show()
    print(
        f"nearest camera frame: {frame_times[onset_frame_index] * 1e3:.2f} ms; "
        f"future vacuum seeds: R={R_STARTUP_REF:.3f} m and R_ECR={R_ECR:.3f} m"
    )


## 10. Comparative startup analysis

Do not duplicate the single-shot notebook for every discharge. Load a list/ODC, compute the same
event/features per shot, and compare both absolute time and a breakdown-aligned convention. Useful
columns are: breakdown time, peak \(I_p\), prefill pressure, representative TF/PF settings,
\(V_{\rm loop}\), reference \(B_z\), decay index, null/Lloyd metrics, and EC-on/off once #165 maps
a historically verified launcher signal.

The example is intentionally commented so the packaged tutorial remains offline.


In [ ]:
# shots = [39915, 41672, 45531]
# ods_list = vaft.database.load(shots)
#
# for case in ods_list:
#     t_bd = float(vaft.omas.find_breakdown_onset(case))
#     print(case["dataset_description.data_entry.pulse"], t_bd)
#
# Many canonical vaft.omas.plot_* adapters accept an ODC or list[ODS] directly.
# For event-aligned comparisons, use the VAFT breakdown time convention rather
# than manually shifting each diagnostic array in notebook code.


## Independent exercise — one-page startup report

Use the actual VEST experiment log and the VAFT database.

1. Choose an experiment date with a startup scan or clearly different operating conditions.
2. Select **at least 2–3 shots**.
3. Compare the commanded/mapped actuators, prefill, detected breakdown time, vacuum null/field,
   \(V_{\rm loop}\), \(B_z\), decay index and Lloyd/connection-length evidence available for them.
4. Separate observations from reduced-model interpretation, and state missing diagnostics or
   convention uncertainties explicitly.
5. Submit a **one-page report**: experiment question, shots, one compact comparison figure/table,
   physical interpretation, and limitations.


In [ ]:
# Example starting point (lab/database mode):
# shots = [<shot_a>, <shot_b>, <shot_c>]
# cases = vaft.database.load(shots)
# ...


## Takeaways

- Establish the event time early and reuse it.
- Treat the vessel as part of the driven electromagnetic circuit.
- Validate coil + eddy synthetic magnetics before interpreting derived vacuum quantities.
- Keep raw DAQ, calibrated physical sensors and derived physics signals conceptually separate.
- \(V_{\rm loop}\), \(B_z\), decay index and Lloyd/Townsend relations are deliberately reduced
  startup tools; equilibrium and stability analysis come later.
- The tutorial fixes **2.45 GHz** as a visible teaching assumption for resonance geometry only;
  EC power mapping remains a separate data-integration task under #165.
